# Parameter Tuning

Systematic parameter optimization using grid search and Bayesian optimization.

**Learning objectives:**
- Understand parameter impact
- Run grid search
- Implement Bayesian optimization
- Avoid overfitting

**Time**: 30 minutes

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from itertools import product

sys.path.insert(0, str(Path.cwd().parent))

from agent import AgentConfig, AutoPredictAgent

## Single Parameter Sweep

Vary one parameter while holding others constant.

In [ ]:
# Sweep min_edge parameter
min_edges = np.linspace(0.02, 0.15, 10)
results = []

for min_edge in min_edges:
    config = AgentConfig(min_edge=min_edge)
    agent = AutoPredictAgent(config)
    
    # Run backtest (simplified)
    sharpe = 2.0 - abs(min_edge - 0.08) * 5  # Placeholder
    
    results.append(sharpe)
    print(f"min_edge={min_edge:.3f} → Sharpe={sharpe:.2f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(min_edges, results, marker='o', linewidth=2)
plt.xlabel('min_edge')
plt.ylabel('Sharpe Ratio')
plt.title('Sharpe vs min_edge')
plt.grid(alpha=0.3)
plt.show()

best_idx = np.argmax(results)
print(f"\nBest min_edge: {min_edges[best_idx]:.3f} (Sharpe={results[best_idx]:.2f})")

## Grid Search

Exhaustive search over 2D parameter grid.

In [ ]:
# Define parameter ranges
min_edges = [0.03, 0.05, 0.08, 0.10]
aggressive_edges = [0.10, 0.12, 0.15, 0.20]

# Grid search
results = np.zeros((len(min_edges), len(aggressive_edges)))

for i, min_edge in enumerate(min_edges):
    for j, agg_edge in enumerate(aggressive_edges):
        config = AgentConfig(
            min_edge=min_edge,
            aggressive_edge=agg_edge
        )
        
        # Run backtest (placeholder)
        sharpe = 2.0 - abs(min_edge - 0.08) - abs(agg_edge - 0.12)
        results[i, j] = sharpe
        
        print(f"min_edge={min_edge:.2f}, agg_edge={agg_edge:.2f} → Sharpe={sharpe:.2f}")

# Visualize heatmap
plt.figure(figsize=(10, 8))
plt.imshow(results, cmap='viridis', aspect='auto')
plt.colorbar(label='Sharpe Ratio')
plt.xticks(range(len(aggressive_edges)), [f"{x:.2f}" for x in aggressive_edges])
plt.yticks(range(len(min_edges)), [f"{x:.2f}" for x in min_edges])
plt.xlabel('aggressive_edge')
plt.ylabel('min_edge')
plt.title('Grid Search Results')
plt.show()

# Find best
best_i, best_j = np.unravel_index(results.argmax(), results.shape)
print(f"\nBest parameters:")
print(f"  min_edge: {min_edges[best_i]:.2f}")
print(f"  aggressive_edge: {aggressive_edges[best_j]:.2f}")
print(f"  Sharpe: {results[best_i, best_j]:.2f}")

## Bayesian Optimization

Smarter search using Gaussian processes.

In [ ]:
# Simplified Bayesian optimization
# In practice, use scikit-optimize or similar

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

# Initialize with random samples
X_observed = []
y_observed = []

for _ in range(5):
    min_edge = np.random.uniform(0.02, 0.15)
    agg_edge = np.random.uniform(0.08, 0.25)
    
    # Evaluate
    sharpe = 2.0 - abs(min_edge - 0.08) - abs(agg_edge - 0.12) + np.random.normal(0, 0.1)
    
    X_observed.append([min_edge, agg_edge])
    y_observed.append(sharpe)

# Fit Gaussian process
gp = GaussianProcessRegressor(kernel=RBF())
gp.fit(X_observed, y_observed)

# Find next point (simplified)
X_test = [[np.random.uniform(0.02, 0.15), np.random.uniform(0.08, 0.25)] for _ in range(100)]
y_pred, y_std = gp.predict(X_test, return_std=True)

# Choose point with highest upper confidence bound
ucb = y_pred + 2 * y_std
best_idx = np.argmax(ucb)

print(f"Suggested next point:")
print(f"  min_edge: {X_test[best_idx][0]:.3f}")
print(f"  aggressive_edge: {X_test[best_idx][1]:.3f}")
print(f"  Expected Sharpe: {y_pred[best_idx]:.2f} ± {y_std[best_idx]:.2f}")

## Avoiding Overfitting

Use train/validation split.

In [ ]:
print("Best Practices:")
print("\n1. Split data:")
print("   - Train: Optimize parameters")
print("   - Validation: Select best parameters")
print("   - Test: Final evaluation")

print("\n2. Limit parameter count:")
print("   - Tune < 5 parameters at once")
print("   - More parameters = more risk of overfitting")

print("\n3. Walk-forward testing:")
print("   - Train on period 1, test on period 2")
print("   - Roll forward, repeat")
print("   - See BACKTESTING.md for details")

print("\n4. Check degradation:")
in_sample_sharpe = 2.5
out_of_sample_sharpe = 1.8
degradation = (in_sample_sharpe - out_of_sample_sharpe) / in_sample_sharpe
print(f"   - In-sample: {in_sample_sharpe:.2f}")
print(f"   - Out-of-sample: {out_of_sample_sharpe:.2f}")
print(f"   - Degradation: {degradation:.1%}")
if degradation > 0.4:
    print("   - WARNING: High degradation suggests overfitting!")

## Next Steps

- Implement walk-forward testing (see BACKTESTING.md)
- Read LEARNING.md for advanced optimization techniques
- Validate optimized parameters on new data before deployment